# Research Study: End-to-End Voice Deepfake Detection with Light-CNN and LFCC
## ASVspoof 2019 Logical Access (LA) Benchmark

### Abstract and Scientific Background
Neural speech synthesis architectures (Text-to-Speech and Voice Conversion) introduce characteristic high-frequency spectral discontinuities, phase inconsistencies, and vocal tract rigidity during neural vocoding (e.g. WaveNet, WORLD, HiFi-GAN). While standard Mel filterbanks attenuate high-frequency bands to emulate human psychoacoustics, **Linear Frequency Cepstral Coefficients (LFCC)** employ uniformly spaced triangular filterbanks across the full Nyquist range ($0 - 8000\text{ Hz}$). This retains essential forensic artifact cues.

The **Light-CNN** architecture incorporates **Max-Feature-Map (MFM)** activation functions:
$$\text{MFM}(x) = \max(x_{2k-1}, x_{2k})$$
MFM performs competitive feature selection, filtering out stochastic recording noise and preserving deterministic vocoder fingerprint signatures.

This notebook executes a complete end-to-end scientific workflow:
1. Environment configuration and hardware diagnostics
2. Dataset discovery with multi-path resolution
3. Protocol parsing and class imbalance audit
4. Pre-cleaning Exploratory Data Analysis (distributions, waveforms, STFT)
5. Deterministic preprocessing (VAD, pre-emphasis, windowing, normalization)
6. Feature engineering (60-dimensional LFCC extraction)
7. Post-preprocessing representation audit
8. PyTorch dataset construction with SpecAugment and balanced sampling
9. Light-CNN architecture formulation with MFM
10. Focal Loss and Cosine Annealing optimization
11. Epoch-by-epoch training and validation tracking
12. Comprehensive publication-grade diagnostic evaluations (Loss, EER, ROC, PR, Confusion Matrix)
13. Granular attack vulnerability analysis (A01 through A06)
14. Latent representation space visualization via t-SNE
15. Model explainability through LFCC Grad-CAM saliency maps
16. End-to-end file-level inference demonstration

## Step 1: Library Installation and Environment Setup

In [ ]:
import subprocess
subprocess.run(["pip", "install", "soundfile", "librosa", "-q"])

import os, glob, time, json, math, random
import numpy as np, pandas as pd
import soundfile as sf, librosa, scipy.fftpack as fft_
from scipy.ndimage import gaussian_filter1d
import matplotlib.pyplot as plt
from sklearn.metrics import (
    roc_curve, roc_auc_score, precision_recall_curve,
    confusion_matrix, accuracy_score, precision_recall_fscore_support
)
from sklearn.manifold import TSNE

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = device.type == "cuda"
print(f"Compute Device: {device}")
if use_amp:
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

fig_dir = "/kaggle/working/figures"
os.makedirs(fig_dir, exist_ok=True)
print(f"Publication Figures Directory: {fig_dir}")


## Step 2: Dataset Discovery and Path Resolution

In [ ]:
def locate_la_root():
    candidates = [
        "/kaggle/input/asvpoof-2019-dataset/LA",
        "/kaggle/input/datasets/awsaf49/asvpoof-2019-dataset/LA/LA",
        "/kaggle/input/datasets/awsaf49/asvpoof-2019-dataset/LA",
        "/kaggle/input/asvpoof2019-dataset/LA",
        "/kaggle/input/asvpoof-2019/LA"
    ]
    for pattern in ["/kaggle/input/**/ASVspoof2019_LA_cm_protocols", "/kaggle/input/**/ASVspoof2019_LA_train"]:
        for match in glob.glob(pattern, recursive=True):
            candidates.insert(0, os.path.dirname(match))
    for c in candidates:
        if os.path.isdir(c) and os.path.isdir(os.path.join(c, "ASVspoof2019_LA_train")):
            return os.path.abspath(c)
    return "/kaggle/input"

la_root = locate_la_root()

def get_proto_file(root, part, suffix):
    direct = os.path.join(root, "ASVspoof2019_LA_cm_protocols", f"ASVspoof2019.LA.cm.{part}.{suffix}.txt")
    if os.path.isfile(direct):
        return direct
    matches = glob.glob(f"/kaggle/input/**/ASVspoof2019.LA.cm.{part}.{suffix}.txt", recursive=True)
    if matches:
        return matches[0]
    matches = glob.glob(f"/kaggle/input/**/*{part}*.txt", recursive=True)
    for m in matches:
        base = os.path.basename(m).lower()
        if "cm" in base and not base.startswith("._"):
            return m
    return direct

def get_flac_dir(root, part):
    direct = os.path.join(root, f"ASVspoof2019_LA_{part}", "flac")
    if os.path.isdir(direct):
        return direct
    matches = glob.glob(f"/kaggle/input/**/ASVspoof2019_LA_{part}/flac", recursive=True)
    if matches:
        return matches[0]
    matches = glob.glob(f"/kaggle/input/**/ASVspoof2019_LA_{part}", recursive=True)
    for m in matches:
        sub = os.path.join(m, "flac")
        if os.path.isdir(sub):
            return sub
        return m
    return direct

flac_dirs = {
    "train": get_flac_dir(la_root, "train"),
    "dev": get_flac_dir(la_root, "dev"),
    "eval": get_flac_dir(la_root, "eval")
}

proto_files = {
    "train": get_proto_file(la_root, "train", "trn"),
    "dev": get_proto_file(la_root, "dev", "trl"),
    "eval": get_proto_file(la_root, "eval", "trl")
}

print(f"Resolved LA Root: {la_root}")
for k in ["train", "dev", "eval"]:
    p_ok = os.path.isfile(proto_files[k])
    d_ok = os.path.isdir(flac_dirs[k])
    d_count = len(os.listdir(flac_dirs[k])) if d_ok else 0
    print(f"  [{k.upper()}] Protocol: {'OK' if p_ok else 'MISSING'} ({proto_files[k]})")
    print(f"          Audio:    {'OK' if d_ok else 'MISSING'} ({d_count:,} files in {flac_dirs[k]})")

assert os.path.isfile(proto_files["train"]), f"Missing training protocol: {proto_files['train']}"
assert os.path.isfile(proto_files["dev"]), f"Missing dev protocol: {proto_files['dev']}"


## Step 3: Protocol Parsing and Metadata Manifest Construction

In [ ]:
rows = []
for partition, path in proto_files.items():
    if not os.path.exists(path):
        continue
    flac_folder = flac_dirs[partition]
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            spk, aid, env, atk, key = parts[0], parts[1], parts[2], parts[3], parts[4]
            rows.append({
                "speaker_id": spk,
                "audio_id": aid,
                "environment_id": env,
                "attack_id": atk,
                "key": key,
                "is_spoof": 1 if key == "spoof" else 0,
                "partition": partition,
                "file_path": os.path.join(flac_folder, f"{aid}.flac")
            })

manifest = pd.DataFrame(rows)
assert len(manifest) > 0, "Protocol parsing produced 0 records. Check dataset paths."
print(f"Total Parsed Records: {len(manifest):,}")

summary_table = []
for part in ["train", "dev", "eval"]:
    sub = manifest[manifest["partition"] == part]
    if len(sub) == 0:
        continue
    bon = int((sub["key"] == "bonafide").sum())
    spf = int((sub["key"] == "spoof").sum())
    summary_table.append({
        "Partition": part,
        "Total Utterances": len(sub),
        "Bonafide": bon,
        "Spoof": spf,
        "Spoof:Bonafide Ratio": f"{spf / max(bon, 1):.2f}:1"
    })
print(pd.DataFrame(summary_table).to_string(index=False))


## Step 4: Exploratory Data Analysis - Class Distribution and Attack Taxonomy

In [ ]:
plt.figure(figsize=(10, 5))
partitions = ["train", "dev", "eval"]
bon_counts = [int(((manifest["partition"] == p) & (manifest["key"] == "bonafide")).sum()) for p in partitions]
spf_counts = [int(((manifest["partition"] == p) & (manifest["key"] == "spoof")).sum()) for p in partitions]

x_pos = np.arange(len(partitions))
width = 0.35

plt.bar(x_pos - width / 2, bon_counts, width, label="Bonafide (Authentic)", color="steelblue")
plt.bar(x_pos + width / 2, spf_counts, width, label="Spoof (Synthetic)", color="firebrick")

plt.xlabel("Dataset Partition", fontsize=12)
plt.ylabel("Number of Utterances", fontsize=12)
plt.title("ASVspoof 2019 LA Class Distribution Across Partitions", fontsize=13)
plt.xticks(x_pos, [p.upper() for p in partitions], fontsize=11)
plt.ylim(0, max(spf_counts) * 1.12)
plt.legend(fontsize=11)
plt.grid(axis="y", linestyle="--", alpha=0.4)

offset = max(spf_counts) * 0.015
for i in range(len(partitions)):
    plt.text(x_pos[i] - width / 2, bon_counts[i] + offset, f"{bon_counts[i]:,}", ha="center", fontsize=9)
    plt.text(x_pos[i] + width / 2, spf_counts[i] + offset, f"{spf_counts[i]:,}", ha="center", fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "01_class_distribution.png"), dpi=300, bbox_inches="tight")
plt.show()


## Step 5: Exploratory Data Analysis - Raw Waveform and Spectral Domain Artifacts

In [ ]:
sample_bon = manifest[(manifest["partition"] == "train") & (manifest["key"] == "bonafide")].iloc[0]
sample_spf = manifest[(manifest["partition"] == "train") & (manifest["key"] == "spoof")].iloc[0]

y_bon_raw, sr_bon = sf.read(sample_bon["file_path"])
y_spf_raw, sr_spf = sf.read(sample_spf["file_path"])
if y_bon_raw.ndim > 1:
    y_bon_raw = y_bon_raw.mean(axis=1)
if y_spf_raw.ndim > 1:
    y_spf_raw = y_spf_raw.mean(axis=1)

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
time_bon = np.linspace(0, len(y_bon_raw) / sr_bon, len(y_bon_raw))
time_spf = np.linspace(0, len(y_spf_raw) / sr_spf, len(y_spf_raw))

axes[0].plot(time_bon, y_bon_raw, color="steelblue", lw=0.7)
axes[0].set_title(f"Raw Authentic Speech (Bonafide: {sample_bon['audio_id']})", fontsize=11)
axes[0].set_ylabel("Amplitude", fontsize=10)
axes[0].grid(True, alpha=0.3)

axes[1].plot(time_spf, y_spf_raw, color="firebrick", lw=0.7)
axes[1].set_title(f"Raw Synthetic Speech (Spoof {sample_spf['attack_id']}: {sample_spf['audio_id']})", fontsize=11)
axes[1].set_xlabel("Time (seconds)", fontsize=10)
axes[1].set_ylabel("Amplitude", fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "02_raw_waveform_comparison.png"), dpi=300, bbox_inches="tight")
plt.show()

stft_bon = np.abs(librosa.stft(y_bon_raw[:min(len(y_bon_raw), 48000)], n_fft=1024, hop_length=256))
stft_spf = np.abs(librosa.stft(y_spf_raw[:min(len(y_spf_raw), 48000)], n_fft=1024, hop_length=256))

db_bon = librosa.amplitude_to_db(stft_bon, ref=np.max)
db_spf = librosa.amplitude_to_db(stft_spf, ref=np.max)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
im0 = axes[0].imshow(db_bon, origin="lower", aspect="auto", cmap="inferno", vmin=-80, vmax=0)
axes[0].set_title("STFT Power Spectrogram: Bonafide", fontsize=11)
axes[0].set_xlabel("Time Frames", fontsize=10)
axes[0].set_ylabel("Frequency Bins (Linear)", fontsize=10)
fig.colorbar(im0, ax=axes[0], format="%+2.0f dB")

im1 = axes[1].imshow(db_spf, origin="lower", aspect="auto", cmap="inferno", vmin=-80, vmax=0)
axes[1].set_title(f"STFT Power Spectrogram: Spoof ({sample_spf['attack_id']})", fontsize=11)
axes[1].set_xlabel("Time Frames", fontsize=10)
axes[1].set_ylabel("Frequency Bins (Linear)", fontsize=10)
fig.colorbar(im1, ax=axes[1], format="%+2.0f dB")

plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "03_raw_spectrogram_artifacts.png"), dpi=300, bbox_inches="tight")
plt.show()


## Step 6: Audio Preprocessing Pipeline - VAD, Pre-Emphasis and Normalization

In [ ]:
def load_raw_audio(path, target_sr=16000):
    y, orig_sr = sf.read(path)
    if y.ndim > 1:
        y = y.mean(axis=1)
    y = y.astype(np.float32)
    if orig_sr != target_sr:
        y = librosa.resample(y, orig_sr=orig_sr, target_sr=target_sr)
    return y

def preprocess_audio(y, is_train=False, target_len=64000, alpha=0.97, top_db=40):
    y = np.concatenate([[y[0]], y[1:] - alpha * y[:-1]])
    intervals = librosa.effects.split(y=y, top_db=top_db)
    if len(intervals):
        trimmed = np.concatenate([y[s:e] for s, e in intervals])
        if len(trimmed) > 1000:
            y = trimmed
    n_samples = len(y)
    if n_samples >= target_len:
        start = np.random.randint(0, n_samples - target_len + 1) if is_train else (n_samples - target_len) // 2
        y = y[start:start + target_len]
    else:
        y = np.pad(y, (0, target_len - n_samples), mode="wrap")
    return y / (np.max(np.abs(y)) + 1e-7)

raw_sample = load_raw_audio(sample_bon["file_path"])
proc_sample = preprocess_audio(raw_sample, is_train=False)

fig, axes = plt.subplots(2, 1, figsize=(12, 5))
axes[0].plot(raw_sample, color="gray", lw=0.6)
axes[0].set_title(f"Before Preprocessing: Length = {len(raw_sample):,} samples", fontsize=11)
axes[0].set_ylabel("Amplitude", fontsize=10)
axes[0].grid(True, alpha=0.3)

axes[1].plot(proc_sample, color="navy", lw=0.6)
axes[1].set_title(f"After Preprocessing (Pre-Emphasis + VAD + 4.0s Window): Length = {len(proc_sample):,} samples", fontsize=11)
axes[1].set_xlabel("Sample Index", fontsize=10)
axes[1].set_ylabel("Normalized Amplitude", fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "04_preprocessing_stages.png"), dpi=300, bbox_inches="tight")
plt.show()


## Step 7: Feature Engineering - Linear Frequency Cepstral Coefficients (LFCC)

In [ ]:
def extract_lfcc(y, sr=16000, n_fft=1024, hop_length=256, n_ceps=20, max_frames=251):
    stft = librosa.stft(y, n_fft=n_fft, hop_length=hop_length, center=True)
    power_spec = np.abs(stft) ** 2
    n_bins = power_spec.shape[0]
    filterbank = np.zeros((n_ceps, n_bins), dtype=np.float32)
    points = np.linspace(0, n_bins - 1, n_ceps + 2, dtype=int)
    for i in range(n_ceps):
        lo, mid, hi = points[i], points[i + 1], points[i + 2]
        if mid > lo:
            filterbank[i, lo:mid] = np.linspace(0, 1, mid - lo)
        if hi > mid:
            filterbank[i, mid:hi] = np.linspace(1, 0, hi - mid)
    linear_energies = np.maximum(filterbank @ power_spec, 1e-8)
    static = fft_.dct(np.log(linear_energies), type=2, axis=0, norm="ortho")[:n_ceps]
    delta1 = librosa.feature.delta(static, order=1)
    delta2 = librosa.feature.delta(static, order=2)
    features = np.vstack([static, delta1, delta2]).astype(np.float32)
    if features.shape[1] >= max_frames:
        features = features[:, :max_frames]
    else:
        features = np.pad(features, ((0, 0), (0, max_frames - features.shape[1])), mode="edge")
    return features

test_row = manifest[manifest["partition"] == "train"].iloc[0]
test_audio = preprocess_audio(load_raw_audio(test_row["file_path"]), is_train=False)
test_lfcc = extract_lfcc(test_audio)
print(f"Verified Processed Audio Shape: {test_audio.shape} (Expected: (64000,))")
print(f"Verified LFCC Representation:   {test_lfcc.shape} (Expected: (60, 251))")
assert test_audio.shape == (64000,) and test_lfcc.shape == (60, 251)


## Step 8: Preprocessing Audit - Feature Representation Before vs After

In [ ]:
bon_lfcc = extract_lfcc(preprocess_audio(load_raw_audio(sample_bon["file_path"]), is_train=False))
spf_lfcc = extract_lfcc(preprocess_audio(load_raw_audio(sample_spf["file_path"]), is_train=False))

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

im0 = axes[0].imshow(bon_lfcc, origin="lower", aspect="auto", cmap="viridis")
axes[0].set_title(f"LFCC Feature Map: Bonafide ({sample_bon['audio_id']})", fontsize=11)
axes[0].set_ylabel("LFCC Coefficients (Static + Delta + Delta-Delta)", fontsize=9)
axes[0].axhline(20, color="white", linestyle="--", alpha=0.5)
axes[0].axhline(40, color="white", linestyle="--", alpha=0.5)
fig.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(spf_lfcc, origin="lower", aspect="auto", cmap="viridis")
axes[1].set_title(f"LFCC Feature Map: Spoof ({sample_spf['attack_id']}: {sample_spf['audio_id']})", fontsize=11)
axes[1].set_xlabel("Time Frames (256 hop length = 16ms)", fontsize=10)
axes[1].set_ylabel("LFCC Coefficients (Static + Delta + Delta-Delta)", fontsize=9)
axes[1].axhline(20, color="white", linestyle="--", alpha=0.5)
axes[1].axhline(40, color="white", linestyle="--", alpha=0.5)
fig.colorbar(im1, ax=axes[1])

plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "05_lfcc_feature_comparison.png"), dpi=300, bbox_inches="tight")
plt.show()


## Step 9: PyTorch Dataset with SpecAugment and Balanced Sampling

In [ ]:
class LFCCDataset(Dataset):
    def __init__(self, df, is_train=False):
        self.df = df.reset_index(drop=True)
        self.is_train = is_train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            raw = load_raw_audio(row["file_path"])
            proc = preprocess_audio(raw, is_train=self.is_train)
            feat = extract_lfcc(proc)
        except Exception:
            feat = np.zeros((60, 251), dtype=np.float32)

        if self.is_train:
            if np.random.rand() < 0.5:
                w_t = np.random.randint(1, 31)
                t_0 = np.random.randint(0, max(1, 251 - w_t))
                feat[:, t_0:t_0 + w_t] = feat.mean()
            if np.random.rand() < 0.5:
                w_f = np.random.randint(1, 11)
                f_0 = np.random.randint(0, max(1, 60 - w_f))
                feat[f_0:f_0 + w_f, :] = feat.mean()

        tensor_x = torch.from_numpy(feat).unsqueeze(0)
        tensor_y = torch.tensor(int(row["is_spoof"]), dtype=torch.long)
        return tensor_x, tensor_y

train_df = manifest[manifest["partition"] == "train"].reset_index(drop=True)
dev_df = manifest[manifest["partition"] == "dev"].reset_index(drop=True)

train_dataset = LFCCDataset(train_df, is_train=True)
dev_dataset = LFCCDataset(dev_df, is_train=False)

train_targets = train_df["is_spoof"].values
class_counts = np.bincount(train_targets)
class_weights = 1.0 / class_counts
sample_weights = torch.FloatTensor(class_weights[train_targets])
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

batch_size = 128
train_loader = DataLoader(train_dataset, batch_size=batch_size, sampler=sampler, num_workers=2, pin_memory=True)
dev_loader = DataLoader(dev_dataset, batch_size=batch_size * 2, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train Batches per Epoch: {len(train_loader):,}")
print(f"Dev Batches per Epoch:   {len(dev_loader):,}")


## Step 10: Light-CNN Architecture with Max-Feature-Map (MFM)

In [ ]:
class MFM(nn.Module):
    def forward(self, x):
        c1, c2 = torch.split(x, x.size(1) // 2, dim=1)
        return torch.max(c1, c2)

class LightCNN(nn.Module):
    def __init__(self, in_channels=1, num_classes=2, dropout=0.3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=5, stride=1, padding=2),
            MFM(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(32, 64, kernel_size=1),
            MFM(),
            nn.BatchNorm2d(32),
            nn.Conv2d(32, 96, kernel_size=3, stride=1, padding=1),
            MFM(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.BatchNorm2d(48),
            nn.Conv2d(48, 96, kernel_size=1),
            MFM(),
            nn.BatchNorm2d(48),
            nn.Conv2d(48, 128, kernel_size=3, stride=1, padding=1),
            MFM(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(64, 128, kernel_size=1),
            MFM(),
            nn.BatchNorm2d(64),
            nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1),
            MFM(),
            nn.BatchNorm2d(32),
            nn.Conv2d(32, 64, kernel_size=1),
            MFM(),
            nn.BatchNorm2d(32),
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            MFM(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.AdaptiveAvgPool2d(1)
        )
        self.fc_latent = nn.Linear(32, 64)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            self.fc_latent,
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, num_classes)
        )

    def extract_latent(self, x):
        feat = self.features(x)
        feat_flat = torch.flatten(feat, 1)
        return self.fc_latent(feat_flat)

    def forward(self, x):
        return self.classifier(self.features(x))

model = LightCNN().to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total Trainable Parameters: {total_params:,}")

with torch.no_grad():
    dummy_input = torch.zeros(2, 1, 60, 251).to(device)
    dummy_out = model(dummy_input)
    dummy_latent = model.extract_latent(dummy_input)
    print(f"Logits Output Shape: {dummy_out.shape} (Expected: (2, 2))")
    print(f"Latent Output Shape: {dummy_latent.shape} (Expected: (2, 64))")


## Step 11: Focal Loss Function and Optimization Setup

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0, label_smoothing=0.05):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.label_smoothing = label_smoothing

    def forward(self, logits, targets):
        ce_loss = F.cross_entropy(logits, targets, reduction="none", label_smoothing=self.label_smoothing)
        p_t = torch.exp(-ce_loss)
        alpha_factor = torch.where(targets == 1, self.alpha, 1.0 - self.alpha)
        focal_loss = alpha_factor * ((1.0 - p_t) ** self.gamma) * ce_loss
        return focal_loss.mean()

def calculate_eer(y_true, y_score):
    fpr, tpr, thresholds = roc_curve(y_true, y_score, pos_label=1)
    fnr = 1 - tpr
    idx = np.nanargmin(np.abs(fpr - fnr))
    eer_val = float((fpr[idx] + fnr[idx]) / 2)
    optimal_thresh = float(np.clip(thresholds[idx], 0.0, 1.0))
    return eer_val, optimal_thresh

def evaluate_network(model, loader, device, use_amp):
    model.eval()
    scores_list, targets_list = [], []
    with torch.no_grad():
        for x_batch, y_batch in loader:
            x_batch = x_batch.to(device)
            with torch.amp.autocast(device_type=device.type, enabled=use_amp):
                probs = torch.softmax(model(x_batch), dim=1)[:, 1]
            scores_list.append(probs.cpu().numpy())
            targets_list.append(y_batch.numpy())
    scores = np.concatenate(scores_list)
    targets = np.concatenate(targets_list)
    eer, thresh = calculate_eer(targets, scores)
    auc = float(roc_auc_score(targets, scores))
    return eer, thresh, auc, scores, targets


## Step 12: Model Training and Validation Loop

In [ ]:
epochs = 30
lr_init = 1e-3
lr_min = 1e-6
weight_decay = 1e-4

criterion = FocalLoss(alpha=0.75, gamma=2.0, label_smoothing=0.05)
optimizer = torch.optim.AdamW(model.parameters(), lr=lr_init, weight_decay=weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=lr_min)
scaler = torch.amp.GradScaler(device=device.type, enabled=use_amp)

save_dir = "/kaggle/working"
os.makedirs(save_dir, exist_ok=True)
best_checkpoint_path = os.path.join(save_dir, "light_cnn_lfcc_best.pth")
history_path = os.path.join(save_dir, "light_cnn_lfcc_history.json")

best_eer = float("inf")
best_auc = 0.0
best_thresh = 0.5
training_history = []

print(f"Commencing Training: {epochs} Epochs | Accelerator: {device} | AMP: {use_amp}")
print("-" * 80)

for epoch in range(1, epochs + 1):
    start_time = time.time()
    model.train()
    running_loss = 0.0

    for x_batch, y_batch in train_loader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(device_type=device.type, enabled=use_amp):
            logits = model(x_batch)
            loss = criterion(logits, y_batch)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * x_batch.size(0)

    epoch_train_loss = running_loss / len(train_dataset)
    scheduler.step(epoch)

    val_eer, val_thresh, val_auc, _, _ = evaluate_network(model, dev_loader, device, use_amp)
    elapsed = time.time() - start_time

    record = {
        "epoch": epoch,
        "train_loss": round(epoch_train_loss, 5),
        "val_eer": round(val_eer, 5),
        "val_auc": round(val_auc, 5),
        "val_thresh": round(val_thresh, 5),
        "time_sec": round(elapsed, 1)
    }
    training_history.append(record)

    is_best = val_eer < best_eer
    if is_best:
        best_eer = val_eer
        best_auc = val_auc
        best_thresh = val_thresh
        torch.save({
            "epoch": epoch,
            "state_dict": model.state_dict(),
            "eer": best_eer,
            "auc": best_auc,
            "threshold": best_thresh,
            "model_architecture": "LightCNN-LFCC"
        }, best_checkpoint_path)

    flag = " [NEW BEST]" if is_best else ""
    print(f"Epoch [{epoch:02d}/{epochs}] | Loss: {epoch_train_loss:.4f} | Dev EER: {val_eer * 100:.2f}% | Dev AUC: {val_auc:.4f} | Time: {elapsed:.0f}s{flag}")

with open(history_path, "w", encoding="utf-8") as f:
    json.dump(training_history, f, indent=2)

print("-" * 80)
print(f"Training Complete. Optimal Dev EER: {best_eer * 100:.2f}% | Dev AUC: {best_auc:.4f} | Threshold: {best_thresh:.4f}")
print(f"Best Model Checkpoint Saved: {best_checkpoint_path}")


## Step 13: Training Loss Progression Curve

In [ ]:
ep_list = [h["epoch"] for h in training_history]
losses = [h["train_loss"] for h in training_history]

plt.figure(figsize=(8, 5))
plt.plot(ep_list, losses, color="steelblue", lw=2, marker="o", ms=4, label="Focal Training Loss")
plt.xlabel("Epoch", fontsize=11)
plt.ylabel("Loss Value", fontsize=11)
plt.title("Light-CNN Training Loss Trajectory across 30 Epochs", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.4)
plt.legend(fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "06_training_loss_curve.png"), dpi=300, bbox_inches="tight")
plt.show()


## Step 14: Development Equal Error Rate (EER) Progression Curve

In [ ]:
eers = [h["val_eer"] * 100 for h in training_history]

plt.figure(figsize=(8, 5))
plt.plot(ep_list, eers, color="firebrick", lw=2, marker="s", ms=4, label="Dev EER (%)")
plt.axhline(min(eers), color="gray", linestyle="--", label=f"Lowest EER: {min(eers):.2f}%")
plt.xlabel("Epoch", fontsize=11)
plt.ylabel("Equal Error Rate (%)", fontsize=11)
plt.title("Development Set Equal Error Rate (EER) Progression", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.4)
plt.legend(fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "07_dev_eer_curve.png"), dpi=300, bbox_inches="tight")
plt.show()


## Step 15: Receiver Operating Characteristic (ROC) Curve

In [ ]:
checkpoint = torch.load(best_checkpoint_path, map_location=device)
model.load_state_dict(checkpoint["state_dict"])
_, optimal_thresh, _, final_scores, final_targets = evaluate_network(model, dev_loader, device, use_amp)

fpr, tpr, _ = roc_curve(final_targets, final_scores, pos_label=1)

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, color="darkviolet", lw=2, label=f"Light-CNN (AUC = {checkpoint['auc']:.4f})")
plt.plot([0, 1], [0, 1], color="gray", linestyle=":", label="Random Classifier (AUC = 0.5000)")
plt.scatter([checkpoint['eer']], [1 - checkpoint['eer']], color="crimson", s=60, zorder=5, label=f"EER Operating Point ({checkpoint['eer']*100:.2f}%)")
plt.xlabel("False Positive Rate (FPR)", fontsize=11)
plt.ylabel("True Positive Rate (TPR)", fontsize=11)
plt.title("ROC Curve on ASVspoof 2019 Development Partition", fontsize=12)
plt.legend(loc="lower right", fontsize=10)
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "08_roc_curve.png"), dpi=300, bbox_inches="tight")
plt.show()


## Step 16: Precision-Recall Curve Analysis

In [ ]:
prec, rec, _ = precision_recall_curve(final_targets, final_scores, pos_label=1)

plt.figure(figsize=(7, 6))
plt.plot(rec, prec, color="teal", lw=2, label="Precision-Recall Trajectory")
plt.xlabel("Recall", fontsize=11)
plt.ylabel("Precision", fontsize=11)
plt.title("Precision-Recall Curve on Development Partition", fontsize=12)
plt.legend(loc="lower left", fontsize=10)
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "09_precision_recall_curve.png"), dpi=300, bbox_inches="tight")
plt.show()


## Step 17: Confusion Matrix at Optimal EER Threshold

In [ ]:
predicted_binary = (final_scores >= optimal_thresh).astype(int)
cm = confusion_matrix(final_targets, predicted_binary)

plt.figure(figsize=(6, 5))
plt.imshow(cm, interpolation="nearest", cmap="Blues")
plt.title(f"Confusion Matrix (Threshold = {optimal_thresh:.4f})", fontsize=12)
plt.colorbar()
tick_marks = np.arange(2)
plt.xticks(tick_marks, ["Bonafide (0)", "Spoof (1)"], fontsize=10)
plt.yticks(tick_marks, ["Bonafide (0)", "Spoof (1)"], fontsize=10)

thresh_val = cm.max() / 2.0
for i in range(2):
    for j in range(2):
        color = "white" if cm[i, j] > thresh_val else "black"
        plt.text(j, i, f"{cm[i, j]:,}", ha="center", va="center", color=color, fontsize=12, fontweight="bold")

plt.ylabel("True Ground Truth", fontsize=11)
plt.xlabel("Predicted Class", fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "10_confusion_matrix.png"), dpi=300, bbox_inches="tight")
plt.show()

acc = accuracy_score(final_targets, predicted_binary)
p, r, f1, _ = precision_recall_fscore_support(final_targets, predicted_binary, average="binary")
print(f"Accuracy:  {acc * 100:.2f}%")
print(f"Precision: {p * 100:.2f}%")
print(f"Recall:    {r * 100:.2f}%")
print(f"F1-Score:  {f1:.4f}")


## Step 18: Probability Density Distribution - Class Separation

In [ ]:
bonafide_scores = final_scores[final_targets == 0]
spoof_scores = final_scores[final_targets == 1]

plt.figure(figsize=(9, 5))
plt.hist(bonafide_scores, bins=60, density=True, alpha=0.6, color="steelblue", label=f"Bonafide Scores (N={len(bonafide_scores):,})")
plt.hist(spoof_scores, bins=60, density=True, alpha=0.6, color="firebrick", label=f"Spoof Scores (N={len(spoof_scores):,})")
plt.axvline(optimal_thresh, color="black", linestyle="--", lw=2, label=f"Optimal Decision Boundary ({optimal_thresh:.4f})")

plt.xlabel("Predicted Spoof Posterior Probability", fontsize=11)
plt.ylabel("Probability Density", fontsize=11)
plt.title("Posterior Score Separation Density on ASVspoof 2019 Dev Split", fontsize=12)
plt.legend(fontsize=10)
plt.grid(True, linestyle="--", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "11_score_density_distribution.png"), dpi=300, bbox_inches="tight")
plt.show()


## Step 19: Granular Attack Taxonomy Vulnerability Analysis (A01 through A06)

In [ ]:
dev_eval_df = dev_df.copy()
dev_eval_df["spoof_score"] = final_scores
dev_eval_df["predicted_label"] = predicted_binary

attack_mapping = {
    "A01": "TTS: Neural Acoustic (AR RNN) + WaveNet",
    "A02": "TTS: Neural Acoustic (AR RNN) + WORLD",
    "A03": "TTS: Concatenative Unit Selection",
    "A04": "VC: Formant / Pitch Shifting + STRAIGHT",
    "A05": "VC: Variational Autoencoder (VAE)",
    "A06": "VC: Transfer Function Regression + WORLD"
}

breakdown_data = []
for atk_id in ["A01", "A02", "A03", "A04", "A05", "A06"]:
    sub = dev_eval_df[dev_eval_df["attack_id"] == atk_id]
    if len(sub) == 0:
        continue
    correct = int((sub["predicted_label"] == 1).sum())
    detection_acc = correct / len(sub)
    mean_prob = float(sub["spoof_score"].mean())
    breakdown_data.append({
        "Attack ID": atk_id,
        "Algorithm Family": attack_mapping.get(atk_id, "Unknown"),
        "Total Utterances": len(sub),
        "Detection Accuracy (%)": round(detection_acc * 100, 2),
        "Mean Spoof Score": round(mean_prob, 4)
    })

bonafide_sub = dev_eval_df[dev_eval_df["key"] == "bonafide"]
bon_correct = int((bonafide_sub["predicted_label"] == 0).sum())
bon_acc = bon_correct / len(bonafide_sub)
breakdown_data.append({
    "Attack ID": "Bonafide",
    "Algorithm Family": "Authentic Human Speech (VCTK)",
    "Total Utterances": len(bonafide_sub),
    "Detection Accuracy (%)": round(bon_acc * 100, 2),
    "Mean Spoof Score": round(float(bonafide_sub["spoof_score"].mean()), 4)
})

breakdown_df = pd.DataFrame(breakdown_data)
print(breakdown_df.to_string(index=False))


## Step 20: Attack Detection Accuracy Comparison Chart

In [ ]:
plt.figure(figsize=(10, 5))
attack_labels = breakdown_df["Attack ID"].values
accuracies = breakdown_df["Detection Accuracy (%)"].values
bar_colors = ["firebrick" if a.startswith("A") else "steelblue" for a in attack_labels]

bars = plt.bar(attack_labels, accuracies, color=bar_colors, width=0.55)
plt.xlabel("Attack Algorithm Identifier", fontsize=11)
plt.ylabel("Detection Accuracy (%)", fontsize=11)
plt.title("Model Sensitivity Across ASVspoof 2019 Known Attack Taxonomy", fontsize=12)
plt.ylim(0, 105)
plt.grid(axis="y", linestyle="--", alpha=0.4)

for bar in bars:
    h = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2.0, h + 1.5, f"{h:.1f}%", ha="center", fontsize=9, fontweight="bold")

plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "12_attack_accuracy_breakdown.png"), dpi=300, bbox_inches="tight")
plt.show()


## Step 21: Latent Representation Space Visualization via t-SNE

In [ ]:
sample_indices = []
for atk_id in ["A01", "A02", "A03", "A04", "A05", "A06"]:
    idx_atk = dev_eval_df[dev_eval_df["attack_id"] == atk_id].index.tolist()[:100]
    sample_indices.extend(idx_atk)
bon_idx = dev_eval_df[dev_eval_df["key"] == "bonafide"].index.tolist()[:300]
sample_indices.extend(bon_idx)

sub_df = dev_eval_df.loc[sample_indices].reset_index(drop=True)
subset_ds = LFCCDataset(sub_df, is_train=False)
subset_loader = DataLoader(subset_ds, batch_size=64, shuffle=False)

embeddings_list = []
model.eval()
with torch.no_grad():
    for xb, _ in subset_loader:
        xb = xb.to(device)
        with torch.amp.autocast(device_type=device.type, enabled=use_amp):
            emb = model.extract_latent(xb)
        embeddings_list.append(emb.cpu().numpy())

latent_matrix = np.concatenate(embeddings_list)
print(f"Extracted Latent Embeddings: {latent_matrix.shape}")

tsne = TSNE(n_components=2, perplexity=30, random_state=42)
coords_2d = tsne.fit_transform(latent_matrix)

plt.figure(figsize=(9, 7))
is_bon = sub_df["key"] == "bonafide"
plt.scatter(coords_2d[is_bon, 0], coords_2d[is_bon, 1], color="steelblue", alpha=0.8, s=35, label="Authentic (Bonafide)")

palette = {
    "A01": "firebrick", "A02": "darkorange", "A03": "gold",
    "A04": "forestgreen", "A05": "darkviolet", "A06": "crimson"
}
for atk, col in palette.items():
    mask = sub_df["attack_id"] == atk
    if mask.sum() > 0:
        plt.scatter(coords_2d[mask, 0], coords_2d[mask, 1], color=col, alpha=0.8, s=35, label=f"Spoof {atk}")

plt.xlabel("t-SNE Dimension 1", fontsize=11)
plt.ylabel("t-SNE Dimension 2", fontsize=11)
plt.title("t-SNE 2D Projection of Light-CNN 64-Dimensional Latent Embeddings", fontsize=12)
plt.legend(loc="best", fontsize=9)
plt.grid(True, linestyle="--", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "13_tsne_latent_clusters.png"), dpi=300, bbox_inches="tight")
plt.show()


## Step 22: Model Explainability through LFCC Grad-CAM Saliency Maps

In [ ]:
class GradCAMLFCC:
    def __init__(self, target_model, target_layer):
        self.model = target_model
        self.layer = target_layer
        self.activations = None
        self.gradients = None
        self.layer.register_forward_hook(self.forward_hook)
        self.layer.register_full_backward_hook(self.backward_hook)

    def forward_hook(self, module, inp, out):
        self.activations = out.detach()

    def backward_hook(self, module, grad_in, grad_out):
        self.gradients = grad_out[0].detach()

    def generate(self, input_tensor, target_class=1):
        self.model.zero_grad()
        output = self.model(input_tensor)
        score = output[0, target_class]
        score.backward()
        weights = torch.mean(self.gradients, dim=(2, 3), keepdim=True)
        cam = torch.sum(weights * self.activations, dim=1, keepdim=True)
        cam = F.relu(cam)
        cam = F.interpolate(cam, size=(60, 251), mode="bilinear", align_corners=False)
        cam = cam.squeeze().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam

cam_generator = GradCAMLFCC(model, model.features[25])

demo_spoof_row = dev_df[dev_df["key"] == "spoof"].iloc[0]
raw_demo = load_raw_audio(demo_spoof_row["file_path"])
proc_demo = preprocess_audio(raw_demo, is_train=False)
lfcc_demo = extract_lfcc(proc_demo)
tensor_demo = torch.from_numpy(lfcc_demo).unsqueeze(0).unsqueeze(0).to(device)

saliency_map = cam_generator.generate(tensor_demo, target_class=1)

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

im0 = axes[0].imshow(lfcc_demo, origin="lower", aspect="auto", cmap="viridis")
axes[0].set_title(f"Input LFCC Spectrogram (Spoof {demo_spoof_row['attack_id']}: {demo_spoof_row['audio_id']})", fontsize=11)
axes[0].set_ylabel("LFCC Coefficients (60)", fontsize=10)
fig.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(lfcc_demo, origin="lower", aspect="auto", cmap="gray")
im2 = axes[1].imshow(saliency_map, origin="lower", aspect="auto", cmap="hot", alpha=0.6)
axes[1].set_title("Grad-CAM Saliency Map: Forensic Attentive Frequency Bands and Time Frames", fontsize=11)
axes[1].set_xlabel("Time Frames (251)", fontsize=10)
axes[1].set_ylabel("LFCC Coefficients (60)", fontsize=10)
fig.colorbar(im2, ax=axes[1])

plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "14_gradcam_saliency_heatmap.png"), dpi=300, bbox_inches="tight")
plt.show()


## Step 23: End-to-End Inference Verification on Real Samples

In [ ]:
def predict_audio_file(file_path, target_model, compute_device, threshold):
    target_model.eval()
    raw = load_raw_audio(file_path)
    proc = preprocess_audio(raw, is_train=False)
    feat = extract_lfcc(proc)
    tensor = torch.from_numpy(feat).unsqueeze(0).unsqueeze(0).to(compute_device)
    with torch.no_grad():
        with torch.amp.autocast(device_type=compute_device.type, enabled=use_amp):
            prob = torch.softmax(target_model(tensor), dim=1)[0, 1].item()
    decision = "SPOOF (SYNTHETIC)" if prob >= threshold else "BONAFIDE (AUTHENTIC)"
    confidence = prob if prob >= threshold else 1.0 - prob
    return {
        "file_name": os.path.basename(file_path),
        "decision": decision,
        "spoof_probability": round(prob, 5),
        "confidence": f"{confidence * 100:.2f}%"
    }

bonafide_test_sample = dev_df[dev_df["key"] == "bonafide"].iloc[0]["file_path"]
spoof_test_sample = dev_df[dev_df["key"] == "spoof"].iloc[0]["file_path"]

print("Case 1: Ground Truth Authentic Speech")
print(json.dumps(predict_audio_file(bonafide_test_sample, model, device, optimal_thresh), indent=2))
print("\nCase 2: Ground Truth Deepfake Speech")
print(json.dumps(predict_audio_file(spoof_test_sample, model, device, optimal_thresh), indent=2))


## Step 24: Research Artifact Summary and Output Verification

In [ ]:
print("Generated Scientific Artifacts in /kaggle/working:")
print("-" * 65)
for root_path, _, files in os.walk(save_dir):
    for f in sorted(files):
        full_fpath = os.path.join(root_path, f)
        rel_fpath = os.path.relpath(full_fpath, save_dir)
        size_kb = os.path.getsize(full_fpath) / 1024
        print(f"  {rel_fpath:<45} | {size_kb:>9.1f} KB")
